# Exploration du dataset FER2013

Vérification de la structure, de la distribution des classes et aperçu des images.

> Prérequis : `python scripts/prepare_data.py` (voir `data/download.md`).

In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
from torchvision import datasets

from src.constants import CLASSES
from src.dataset import compute_class_weights

os.makedirs("results/figures", exist_ok=True)
plt.rcParams["figure.dpi"] = 120

In [ ]:
train_ds = datasets.ImageFolder("data/raw/train")
test_ds = datasets.ImageFolder("data/raw/test")
print("Classes :", train_ds.classes)
print(f"Train : {len(train_ds)} images")
print(f"Test  : {len(test_ds)} images")
print(f"Total : {len(train_ds) + len(test_ds)} images")
assert train_ds.classes == CLASSES, "Ordre des classes inattendu"

In [ ]:
train_counts = Counter([y for _, y in train_ds.samples])
test_counts = Counter([y for _, y in test_ds.samples])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(CLASSES, [train_counts[i] for i in range(len(CLASSES))])
ax[0].set_title("Train")
ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(CLASSES, [test_counts[i] for i in range(len(CLASSES))])
ax[1].set_title("Test")
ax[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig("results/figures/dataset_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (img_path, label) in zip(axes.flat, train_ds.samples[:8]):
    ax.imshow(plt.imread(img_path), cmap="gray")
    ax.set_title(CLASSES[label])
    ax.axis("off")
fig.suptitle("Exemples d'images (48x48, niveaux de gris)")
fig.tight_layout()
fig.savefig("results/figures/dataset_samples.png", bbox_inches="tight")
plt.show()

In [ ]:
total = sum(train_counts.values())
print(f"{'classe':>10s} {'train':>8s} {'part':>7s}")
for i, name in enumerate(CLASSES):
    print(f"{name:>10s} {train_counts[i]:>8d} {train_counts[i] / total:>6.1%}")

weights = compute_class_weights("data/raw")
print("\nPoids de classes (inverse fréquence) :")
for name, w in zip(CLASSES, weights.tolist()):
    print(f"  {name:>10s} : {w:.3f}")
print(f"\nRatio max/min : {weights.max() / weights.min():.1f}x")